# Content Overview
This notebook demonstrates a simple linear regression workflow for FMCG quantity forecasting.

Contents:
1. Load and inspect the FMCG dataset
2. Select the target variable (`VolumePC_sum`)
3. Prepare the time series data for modeling
4. Build a linear regression model
5. Generate predictions
6. Visualize actual vs. predicted values
7. Evaluate forecast quality using error metrics
   - MAE
   - MSE
   - MAPE
   - WAPE
8. Interpret the results

In [ ]:
import pandas as pd
import statsmodels.api as sm
from matplotlib import pyplot

In [ ]:
file_path = "https://raw.github.com/jan-source/ppa_erp_student/main/sessions/1_session/fmcg_data.xlsx"

In [ ]:
data = pd.read_excel(file_path)

In [ ]:
data

### Zeige die ersten 5 Zeilen

In [ ]:
data.head()

### Wähle die Spalte VolumePC_sum aus

In [ ]:
data['VolumePC_sum']

### Wähle zwei Spalten aus

In [ ]:
data[['Product', 'VolumePC_sum', 'TV']]

### Wähle erste Zeile aus

In [ ]:
data.iloc[0]

### Wähle die ersten beiden Zeilen aus

In [ ]:
data.iloc[0:2]

### von den ersten beiden Zeilen, die Spalten VolumePC_sum, TV auswählen

In [ ]:
data.iloc[0:2][["VolumePC_sum", "TV"]]

### Selektiere alle Zeilen in den VolumePC_sum > 100

In [ ]:
data[data["VolumePC_sum"] > 100]

### Selektiere Zeilen in denen das Verkaufsvolumen exakt 100 ist

In [ ]:
data[data['VolumePC_sum'] == 100]

### Selektiere Zeilen in denen das verkaufte Volumen größer als 200 ist und der Verkaufspreise größer 4

In [ ]:
data[(data['VolumePC_sum'] > 200) & (data['pricePerPieceNonPromo'] > 4)]

### Wie viele unterschiedliche Produkte gibt es

In [ ]:
data['Product'].unique()

In [ ]:
len(data['Product'].unique())

### Absatzvolumen pro Produkt

In [ ]:
data.groupby('Product')['VolumePC_sum'].sum()

### Absatzvolumen nach Absatzkanal

In [ ]:
data.groupby('MarketDescription_global')['VolumePC_sum'].sum()

### Absatzvolumen pro Product/Kanal Kombination

In [ ]:
data.groupby(['Product','MarketDescription_global'])['VolumePC_sum'].sum()

In [ ]:
data[(data['Product'] == 'q1106') & 
     (data['MarketDescription_global'] == 'DISCOUNTER')]['VolumePC_sum'].sum()

### Übersicht Verkaufszahlen im Channel DISCOUNTER und für das Produkt q1106

In [ ]:
subset = data[(data['Product'] == 'q1106') & 
     (data['MarketDescription_global'] == 'DISCOUNTER')]

In [ ]:
pyplot.rcParams['figure.figsize'] = [15, 7]

In [ ]:
pyplot.plot(
    subset['Date'],
    subset['VolumePC_sum']
)

### Aggregiert Daten pro Tag über alle Produkte und Channel

In [ ]:
data.groupby('Date')['VolumePC_sum'].sum().plot()

In [ ]:
data.groupby(['MarketDescription_global','Date'])['VolumePC_sum'].sum().plot()

### Lineare Regression Verkaufsmenge (VolumePC_sum) durch Preis (pricePerPieceNonPromo) erklären

In [ ]:
import statsmodels.api as sm

### Filter Daten für Produkt (q1152) und Kanal (DRUGSTORES)

In [ ]:
subset_q1152_drugstores = data[(data['Product'] == 'q1152') & (data['MarketDescription_global'] == 'DRUGSTORES')]

In [ ]:
subset_q1152_drugstores.shape

### Model definieren

In [ ]:
mod_q1152_drugstores = sm.OLS(subset_q1152_drugstores['VolumePC_sum'], 
                              subset_q1152_drugstores['pricePerPieceNonPromo'])

### Model trainieren

In [ ]:
mod_q1152_drugstores.fit()

In [ ]:
mod_q1152_drugstores.fit().summary()

### Berücksichtigung Achsenabschnitt

In [ ]:
x = subset_q1152_drugstores['pricePerPieceNonPromo']

### Hinzufügen der Konstanten / Achsenabschnitt

In [ ]:
x = sm.add_constant(x)

In [ ]:
x

In [ ]:
mod_q1152_drugstores_with_constant = sm.OLS(subset_q1152_drugstores['VolumePC_sum'], x)

In [ ]:
mod_q1152_drugstores_with_constant.fit().summary()

### Model mit zusätzlichen Variablen:  'isPromoPeriod',  'TV', 'BB', 'DI'

In [ ]:
x = subset_q1152_drugstores[['pricePerPieceNonPromo', 'isPromoPeriod', 'TV', 'BB', 'DI']]

In [ ]:
x = sm.add_constant(x)

In [ ]:
x

In [ ]:
x = x.astype(float)

In [ ]:
mod_q1152_drugstores_with_constant = sm.OLS(subset_q1152_drugstores['VolumePC_sum'], x)

In [ ]:
mod_q1152_drugstores_with_constant.fit().summary()

## Model für 'q1106' im Channel 'DISCOUNTER'

In [ ]:
subset = data[(data['Product'] == 'q1106') & 
     (data['MarketDescription_global'] == 'DISCOUNTER')]

In [ ]:
x = subset[['pricePerPieceNonPromo', 'isPromoPeriod', 'TV', 'BB', 'DI']]

In [ ]:
x = sm.add_constant(x)

In [ ]:
x = x.astype(float)

In [ ]:
calculated_model = sm.OLS(subset['VolumePC_sum'], x).fit()

In [ ]:
calculated_model.summary()

### Nutze Model für Prognose

In [ ]:
predictions = calculated_model.predict(x)

In [ ]:
pyplot.plot(subset['Date'], subset['VolumePC_sum'])
pyplot.plot(subset['Date'], predictions, c='orange')


### Qualität der Vorhersage messen

In [ ]:
sum(subset['VolumePC_sum'] - predictions)

In [ ]:
sum(abs(subset['VolumePC_sum'] - predictions))

### Durchschnittlicher absolute Fehler (MAE):
$MAE=\frac{1}{N} {\sum_{t=1}^{N}{|{y_t}}-\hat{y_{t}}|}$

In [ ]:
sum(abs(subset['VolumePC_sum'] - predictions)) / subset.shape[0]

### Durchschnittlicher quadrierter Fehler (MSE):
$MSE=\frac{1}{N}\sum_{t=1}^{N}(y_t-\hat{y}_t)^2$


In [ ]:
sum((subset['VolumePC_sum'] - predictions) ** 2) / subset.shape[0]

### Durschnittlicher prozentualer Fehler (MAE)
$MAE=\frac{1}{N} \sum_{t=1}^{N}{\frac{{y_t}-\hat{y_{t}}}{y_t}}$

In [ ]:
sum((subset['VolumePC_sum'] - predictions)/subset['VolumePC_sum']) / subset.shape[0]

### Durschnittlicher absoluter prozentualer Fehler (MAPE)
$MAPE=\frac{1}{N} \sum_{t=1}^{N}{\frac{|{y_t}-\hat{y_{t}}|}{y_t}}$

In [ ]:
sum(abs((subset['VolumePC_sum'] - predictions)/subset['VolumePC_sum'])) / subset.shape[0]

### Mittlerer gewichteter absoluter prozentualer Fehler (WAPE)
$WMAPE=\left(\frac{\sum_{t=1}^{N}{|{y_t}}-\hat{y_{t}}|}{\sum_{t=1}^{N}{|y_{t}}|}\right)$

In [ ]:
(sum(abs((subset['VolumePC_sum'] - predictions)))/sum(abs(subset['VolumePC_sum'])))

### Funktion um die Berechnung und Auswertung des Models zusammenzufassen

In [ ]:
def model_trainieren(produkt, kanal, daten):
    subset = daten[(daten['Product'] == produkt) & 
        (daten['MarketDescription_global'] == kanal)]

    # Prüfen, ob genügend Daten vorhanden sind
    if subset.shape[0] > 30:
        # Feature definieren
        x = subset[['pricePerPieceNonPromo', 
                    'isPromoPeriod', 'TV', 'BB', 'DI', 
                    'WeightedDistribution_max']]
        
        # Konstante hinzufügen für den Achsenabschnitt
        x = sm.add_constant(x)  

        # Zielvariable definieren
        y = subset['VolumePC_sum']
        
        # Lineares Regressionsmodell trainieren
        model = sm.OLS(y, x).fit()

        # Calculate Predictions
        predictions = model.predict(x)
        mae = sum(abs(subset['VolumePC_sum'] - predictions)) / subset.shape[0]
        mape = sum(abs((subset['VolumePC_sum'] - predictions)/subset['VolumePC_sum'])) / subset.shape[0]

        return {'status': 'ok', 'model_summary': model.summary(), 'mae': mae, 'mape': mape}
    else:
        return {'status': 'error', 'message': "zu wenig Daten"}


#### Testen der Funktion

In [ ]:
model_trainieren('q1110', 'DRUGSTORES', data)

### Berechne alle Produkt/Kanal - Kombination

In [ ]:

results = []

for i in data['Product'].unique():
    for j in data['MarketDescription_global'].unique():
        result = model_trainieren(i, j, data)      

        # Überprüfen, ob das Modell erfolgreich trainiert wurde
        if result['status'] != 'ok':
            results.append({'product': i, 
                            'channel': j, 
                            'status': result['status'], 
                            'mape': None, 
                            'mae': None})
        else:
            results.append({'product': i, 
                            'channel': j, 
                            'status': result['status'], 
                            'mape': result['mape'],
                            'mae': result['mae']})

### Übersicht der Ergebnisse

In [ ]:
pd.DataFrame(results).sort_values(by='mape')

#### Exportieren der Ergebnisse in Excel

In [ ]:
pd.DataFrame(results).sort_values(by='mape').to_excel('output/results.xlsx', index=False)